# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haroonrana330/flyrank-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Two paper findings + my methodology questions

### Finding 1 — Content Lifecycle: Growing vs Declining

The paper reports that growing pages were younger on average than declining pages: about 185 days versus 228 days. Word count was almost the same between the two groups, so the paper highlights age as the clearer observed difference.

My methodology question is: how is the growth/decline label defined, and does the comparison support the claim being made? The paper defines trend direction using the recent 30 days compared with the previous 30 days, with pages above 10% growth classified as up and pages below 10% classified as down. The comparison is useful for describing a pattern in this dataset, but it does not by itself prove that older content causes decline because other factors can be related to both age and performance.

### Finding 2 — Click Capture by Position Tier

The paper reports that weighted CTR decreases as ranking position gets worse. The reported weighted CTR is highest for the top 3 positions and much lower for pages deep in the results.

My methodology question is: does the validation design support a descriptive relationship, or a causal claim? The paper's methodology supports this as an observed relationship in the studied portfolio, but it should not be interpreted as proof that changing position alone causes a specific amount of CTR improvement. Position, query type, title, description, intent, and other factors can all affect click behavior.

### Overall methodology takeaway

I am treating both findings as observed patterns rather than causal claims. The paper itself states that it is a pattern study and that the main findings rely on direct comparisons. This is the same standard I will apply to my own model: use held-out data and an honest split, then describe the results as measured or directional evidence rather than proof of causation.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1: simple checks for the paper findings

paper_findings = {
    "Finding 1": {
        "topic": "Content lifecycle",
        "main_observation": "Growing pages were younger on average than declining pages.",
        "methodology_question": "Does the growth/decline definition support a descriptive comparison without implying causation?"
    },
    "Finding 2": {
        "topic": "CTR by position tier",
        "main_observation": "Weighted CTR was higher for better ranking positions.",
        "methodology_question": "Does the comparison support an observed relationship rather than a causal claim?"
    }
}

for name, details in paper_findings.items():
    print(name)
    print("Observation:", details["main_observation"])
    print("Methodology question:", details["methodology_question"])
    print()

Finding 1
Observation: Growing pages were younger on average than declining pages.
Methodology question: Does the growth/decline definition support a descriptive comparison without implying causation?

Finding 2
Observation: Weighted CTR was higher for better ranking positions.
Methodology question: Does the comparison support an observed relationship rather than a causal claim?



## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## 2. My model under an honest split (before/after)

I compare two validation designs using the same target and feature set.

The "before" result uses a random row-level split. This can place pages from the same client in both training and testing, which may make the evaluation look more optimistic.

The "after" result uses a grouped split by client_id. This keeps each client's pages together, so the test set contains clients that were not used for training.

The grouped split is the more honest validation design for this task because the model should be useful on clients it has not already seen. I will compare the Random Forest ROC-AUC under both splits and treat the grouped result as the more conservative estimate.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Load the dataset
url = "https://raw.githubusercontent.com/haroonrana330/flyrank-ml/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Create the target used in Week 5
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

# Features used in Week 5
feature_cols = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

feature_cols = [c for c in feature_cols if c in df.columns]

# Clean numeric features
for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].median())

X = df[feature_cols]
y = df["is_declining_label"]
groups = df["client_id"]

# --------------------------------------------------
# BEFORE: random row-level split
# --------------------------------------------------

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

rf_random = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    random_state=42
)

rf_random.fit(X_train_random, y_train_random)

random_probs = rf_random.predict_proba(X_test_random)[:, 1]
random_auc = roc_auc_score(y_test_random, random_probs)

# --------------------------------------------------
# AFTER: grouped split by client
# --------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

rf_group = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    random_state=42
)

rf_group.fit(X_train_group, y_train_group)

group_probs = rf_group.predict_proba(X_test_group)[:, 1]
group_auc = roc_auc_score(y_test_group, group_probs)

# --------------------------------------------------
# Comparison
# --------------------------------------------------

comparison = pd.DataFrame({
    "Validation design": [
        "Random row-level split",
        "Grouped by client"
    ],
    "ROC-AUC": [
        random_auc,
        group_auc
    ]
})

print("HONEST VALIDATION COMPARISON")
print("=" * 60)
print(comparison.to_string(index=False))

print()
print("Random-split test rows:", len(X_test_random))
print("Grouped-split test rows:", len(X_test_group))
print(
    "Unique clients in grouped training:",
    df.iloc[train_idx]["client_id"].nunique()
)
print(
    "Unique clients in grouped testing:",
    df.iloc[test_idx]["client_id"].nunique()
)

print()
if random_auc > group_auc:
    print(
        "The grouped split gives a lower ROC-AUC than the random split, "
        "showing why client-level separation is a more conservative validation design."
    )
else:
    print(
        "The grouped split did not produce a lower ROC-AUC in this run. "
        "The grouped result is still the more appropriate validation design "
        "because it prevents client overlap between training and testing."
    )

HONEST VALIDATION COMPARISON
     Validation design  ROC-AUC
Random row-level split 0.692233
     Grouped by client 0.563739

Random-split test rows: 6000
Grouped-split test rows: 6163
Unique clients in grouped training: 25
Unique clients in grouped testing: 7

The grouped split gives a lower ROC-AUC than the random split, showing why client-level separation is a more conservative validation design.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage audit

The target variable is `is_declining_label`, which is created from `trend_direction`.

Because `trend_direction` is used to create the target, it must not be included as a model feature. I also exclude `trend_pct` because it contains information directly related to the same trend used to define the target.

I checked the final feature set used by the model. Neither `trend_direction`, `trend_pct`, nor `is_declining_label` is included as a feature.

The final model therefore uses independent signals such as search volume, competition, CPC, word count, character count, CTR, average position, engagement rate, scroll rate, and AI traffic percentage.

This reduces the risk of target leakage and makes the model evaluation more trustworthy.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3: Leakage audit

target_col = "is_declining_label"

forbidden_features = {
    "trend_direction",
    "trend_pct",
    target_col
}

leakage_found = [
    feature
    for feature in feature_cols
    if feature in forbidden_features
]

print("TARGET:")
print(target_col)

print()
print("FINAL FEATURES:")
for feature in feature_cols:
    print("-", feature)

print()
print("LEAKAGE CHECK")
print("=" * 60)

if leakage_found:
    print("WARNING: Possible leakage features found:")
    for feature in leakage_found:
        print("-", feature)
else:
    print("PASS: No target-derived leakage features are present.")

print()
print("Explicitly excluded:")
print("- trend_direction")
print("- trend_pct")
print("- is_declining_label")

TARGET:
is_declining_label

FINAL FEATURES:
- search_volume
- competition
- cpc
- word_count
- char_count
- ctr
- avg_position
- engagement_rate
- scroll_rate
- ai_traffic_pct

LEAKAGE CHECK
PASS: No target-derived leakage features are present.

Explicitly excluded:
- trend_direction
- trend_pct
- is_declining_label


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim rewrite

### Original stronger claim

"My Random Forest meaningfully improves prediction of declining pages compared with my baseline."

### Safer rewritten claim

"On this dataset and validation setup, the Random Forest produced a higher measured ROC-AUC than the Week-4 baseline under the tested split. The result is directional evidence that the learned model may rank declining pages more effectively than the baseline, but it should be treated as decision-support evidence rather than proof of general performance on future clients."

### Why I changed the claim

The original wording could sound broader than the evidence supports. The safer version states exactly what was measured, identifies the dataset and validation context, and avoids implying causation or guaranteed future performance.

The model can support prioritization and decision-making, but the result should be re-validated on future or unseen client data before making a stronger performance claim.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4: record the measured result without overstating it

print("SAFE CLAIM CHECK")
print("=" * 60)

print(
    f"Observed random-split ROC-AUC: {random_auc:.4f}"
)

print(
    f"Observed client-grouped ROC-AUC: {group_auc:.4f}"
)

print()
print(
    "Safe interpretation: these are measured results on this dataset "
    "under the stated validation designs."
)

print(
    "Decision-support interpretation: the grouped result is the preferred "
    "estimate for judging performance on unseen clients."
)

print(
    "Causal claim: not established by this experiment."
)

SAFE CLAIM CHECK
Observed random-split ROC-AUC: 0.6922
Observed client-grouped ROC-AUC: 0.5637

Safe interpretation: these are measured results on this dataset under the stated validation designs.
Decision-support interpretation: the grouped result is the preferred estimate for judging performance on unseen clients.
Causal claim: not established by this experiment.


## Self-check

- [x] Section 1 identifies two paper findings and asks constructive methodology questions.
- [x] Section 2 compares a random row-level split with a client-grouped split.
- [x] Section 2 uses the grouped-by-client result as the more honest validation design.
- [x] Section 3 checks the final feature set for target-derived leakage.
- [x] `trend_direction`, `trend_pct`, and `is_declining_label` are not used as model features.
- [x] Section 4 rewrites the model claim using measured, directional, and decision-support language.
- [x] The notebook uses actual computed results rather than invented numbers.
- [x] The notebook should run from top to bottom without errors.
- [x] No client names, private queries, or private URLs are included.
- [x] The completed notebook will be committed under `work/notebooks/w06_validation_audit.ipynb`.
- [x] The GitHub repository URL will be submitted on the ML-09 assignment card.